# 03 · Silver → Gold

Neste notebook é construída a modelagem dimensional (*Star Schema*) da camada Gold para atender às necessidades analíticas de BI e Inteligência de Mercado.
A modelagem é composta por:
- Tabelas Dimensão com chaves substitutas (*Surrogate Keys*) do tipo `BIGINT`;
- Tabelas-Ponte (*Bridge Tables*) para suportar relacionamentos muitos-para-muitos (N:N);
- Tabela Fato com granularidade única por filme lançado;
- Consultas analíticas respondendo às perguntas de negócio do projeto.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "cinedata_analytics"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

print(f"Lendo de: {silver_schema}")
print(f"Gravando em: {gold_schema}")


Lendo de: cinedata_analytics.silver
Gravando em: cinedata_analytics.gold


## 1. Construção das Tabelas Dimensão

São geradas as seguintes dimensões descritivas, utilizando `row_number()` para a criação das chaves substitutas (`sk_`):
- `gold.dim_movies`: Metadados cadastrais e descritivos de cada obra;
- `gold.dim_genres`: Catálogo único de gêneros;
- `gold.dim_people`: Pessoas físicas envolvidas nas produções (Atores, Diretores e Roteiristas);
- `gold.dim_companies`: Catálogo único de estúdios e produtoras;
- `gold.dim_reviews`: Consolidação das avaliações com contagem total e média de nota por filme.

In [0]:
df_info_silver = spark.table(f"{silver_schema}.tb_info_filmes")
window_movie = Window.orderBy("id_filme")

df_dim_movies = (
    df_info_silver
    .withColumn("sk_movie_id", F.row_number().over(window_movie).cast("bigint"))
    .select(
        "sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento",
        "duracao_minutos", "idioma_original", "status_filme", "sinopse"
    )
)
df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

df_generos_silver = spark.table(f"{silver_schema}.tb_generos")
window_genre = Window.orderBy("nome_genero")

df_dim_genres = (
    df_generos_silver
    .select("nome_genero").distinct()
    .withColumn("sk_genre_id", F.row_number().over(window_genre).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)
df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_genres")

df_pe_silver = spark.table(f"{silver_schema}.tb_pessoas_empresas")
window_person = Window.orderBy("nome_pessoa", "tipo_pessoa")

df_dim_people = (
    df_pe_silver
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .distinct()
    .withColumn("sk_person_id", F.row_number().over(window_person).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)
df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")

window_company = Window.orderBy("nome_produtora")

df_dim_companies = (
    df_pe_silver
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .withColumn("sk_company_id", F.row_number().over(window_company).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)
df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_companies")

dim_movies_lk = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")
df_rev_silver = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
window_review = Window.orderBy("sk_movie_id")

df_dim_reviews = (
    df_rev_silver
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    )
    .join(dim_movies_lk, on="id_filme", how="inner")
    .withColumn("sk_review_id", F.row_number().over(window_review).cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)
df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

print("Tudo certo")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tudo certo


## 2. Construção das Tabelas-Ponte (*Bridge Tables*)

Como uma obra cinematográfica pode possuir múltiplos gêneros, múltiplos atores e múltiplos estúdios (e vice-versa), são criadas tabelas-ponte para conectar a tabela de filmes às dimensões periféricas, preservando a integridade do grão da Fato e evitando duplicações métricas:
- `gold.bridge_movie_genre`
- `gold.bridge_movie_person`
- `gold.bridge_movie_company`

In [0]:
dim_genres_lk = spark.table(f"{gold_schema}.dim_genres")
df_bridge_genre = (
    spark.table(f"{silver_schema}.tb_generos").alias("g")
    .join(dim_movies_lk.alias("m"), F.col("g.id_filme") == F.col("m.id_filme"), "inner")
    .join(dim_genres_lk.alias("dg"), F.col("g.nome_genero") == F.col("dg.nome_genero"), "inner")
    .select("m.sk_movie_id", "dg.sk_genre_id")
    .distinct()
)
df_bridge_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")

dim_people_lk = spark.table(f"{gold_schema}.dim_people")
df_bridge_person = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas").alias("pe")
    .filter(F.col("pe.tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(dim_movies_lk.alias("m"), F.col("pe.id_filme") == F.col("m.id_filme"), "inner")
    .join(dim_people_lk.alias("dp"), 
          (F.col("pe.nome_entidade") == F.col("dp.nome_pessoa")) & 
          (F.col("pe.tipo_entidade") == F.col("dp.tipo_pessoa")), "inner")
    .select("m.sk_movie_id", "dp.sk_person_id")
    .distinct()
)
df_bridge_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")

dim_companies_lk = spark.table(f"{gold_schema}.dim_companies")
df_bridge_company = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas").alias("pe")
    .filter(F.col("pe.tipo_entidade") == "Produtora")
    .join(dim_movies_lk.alias("m"), F.col("pe.id_filme") == F.col("m.id_filme"), "inner")
    .join(dim_companies_lk.alias("dc"), F.col("pe.nome_entidade") == F.col("dc.nome_produtora"), "inner")
    .select("m.sk_movie_id", "dc.sk_company_id")
    .distinct()
)
df_bridge_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")

print("As 3 Bridge Tables foram criadas com sucesso")

As 3 Bridge Tables foram criadas com sucesso


## 3. Construção da Tabela Fato (`gold.fact_movies_performance`)

A tabela fato consolida as métricas financeiras (`DECIMAL(18,2)`) e de engajamento do filme.
- **Grão:** Um registro único por filme lançado (`status_filme == 'Lançado'`).
- **Chave Estrangeira:** `sk_movie_id` (apontando para `dim_movies`).

In [0]:
dim_movies_lancados = (
    spark.table(f"{gold_schema}.dim_movies")
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
)

df_fin_silver = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_met_silver = spark.table(f"{silver_schema}.tb_metricas_engajamento")

df_fact_movies = (
    dim_movies_lancados.alias("m")
    .join(df_fin_silver.alias("f"), F.col("m.id_filme") == F.col("f.id_filme"), "left")
    .join(df_met_silver.alias("e"), F.col("m.id_filme") == F.col("e.id_filme"), "left")
    .select(
        F.col("m.sk_movie_id"),
        F.col("f.orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("f.receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        F.col("f.lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        F.col("f.orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("f.receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        F.col("f.lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        F.col("e.popularidade").cast("double").alias("popularidade"),
        F.col("e.nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.col("e.qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        F.col("e.nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.col("e.qtd_votos_imdb").cast("int").alias("qtd_votos_imdb")
    )
)

df_fact_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

print("gold.fact_movies_performance criada com sucesso!")
display(df_fact_movies.filter(F.col("receita_usd").isNotNull()).limit(10))

gold.fact_movies_performance criada com sucesso!


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
149,3000.00,8000.00,5000.00,14907.60,39753.60,24846.00,1.386,0.0,0,6.9,9
157,null,34900000.00,null,null,173425080.00,null,295.179,6.5,31,5.2,null
299,10000.00,3000000.00,2990000.00,49692.00,14907600.00,14857908.00,1.234,10.0,1,8.6,40
380,null,894733.00,null,null,4446107.22,null,11.913,7.361,90,7.5,9409
390,null,30189.00,null,null,150015.18,null,2.218,0.0,0,7.1,2006
394,null,103.00,null,null,511.83,null,0.6,0.0,0,5.7,11
484,null,460000.00,null,null,2285832.00,null,1.102,6.7,3,5.5,136
491,null,4074922.00,null,null,20249102.40,null,1.822,6.938,8,4.7,609
718,4500000.00,72600000.00,68100000.00,22361400.00,360763920.00,338402520.00,1458.514,7.214,973,7.1,224148
720,null,882348.00,null,null,4384563.68,null,21.616,6.455,67,6.8,12672


## Perguntas de Negócio:

### Pergunta 1
Qual é a receita total (em R$) somada de todos os filmes da base?

In [0]:
df_p1 = (
    spark.table(f"{gold_schema}.fact_movies_performance")
    .select(
        F.round(F.sum("receita_brl"), 2).alias("receita_total_brl"),
        F.round(F.sum("receita_usd"), 2).alias("receita_total_usd")
    )
)

print("RECEITA TOTAL SOMADA")
display(df_p1)

RECEITA TOTAL SOMADA


receita_total_brl,receita_total_usd
804350342157.83,161867170200.00


### Pergunta 2
Quais são os 5 filmes com maior popularidade? (Mostrando título e valor de popularidade).

In [0]:
fact = spark.table(f"{gold_schema}.fact_movies_performance").alias("f")
movies = spark.table(f"{gold_schema}.dim_movies").alias("m")

df_p2 = (
    fact
    .join(movies, "sk_movie_id", "inner")
    .filter(F.col("f.popularidade").isNotNull())
    .select(
        F.col("m.titulo"),
        F.col("f.popularidade")
    )
    .orderBy(F.col("f.popularidade").desc())
    .limit(5)
)

print("TOP 5 FILMES COM MAIOR POPULARIDADE")
display(df_p2)

TOP 5 FILMES COM MAIOR POPULARIDADE


titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
La Fellinette,2020.0
The I and S of Lives,2020.0
The Fear Footage 2: Curse of the Tape,2019.0


### Pergunta 3
Quantos filmes cada gênero possui? (Ordenado do maior para o menor volume).

In [0]:
bridge_genre = spark.table(f"{gold_schema}.bridge_movie_genre").alias("bg")
genres = spark.table(f"{gold_schema}.dim_genres").alias("g")

df_p3 = (
    bridge_genre
    .join(genres, "sk_genre_id", "inner")
    .groupBy(F.col("g.nome_genero"))
    .agg(F.countDistinct("bg.sk_movie_id").alias("quantidade_filmes"))
    .orderBy(F.col("quantidade_filmes").desc())
)

print("QUANTIDADE DE FILMES POR GÊNERO")
display(df_p3)

QUANTIDADE DE FILMES POR GÊNERO


nome_genero,quantidade_filmes
Drama,32306
Documentary,19073
Comedy,18630
Thriller,10276
Horror,9728
Romance,7640
Action,6056
Crime,4739
Animation,4473
Science Fiction,3772


### Pergunta 4
Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição no ranking utilizando a função analítica `RANK()`.

In [0]:
janela_ranking = Window.orderBy(F.col("f.receita_usd").desc())

df_p4 = (
    fact
    .join(movies, "sk_movie_id", "inner")
    .filter(F.col("f.receita_usd").isNotNull())
    .withColumn("posicao_ranking", F.rank().over(janela_ranking))
    .select(
        F.col("posicao_ranking"),
        F.col("m.titulo"),
        F.col("f.receita_usd"),
        F.col("f.receita_brl")
    )
    .orderBy("posicao_ranking")
    .limit(10)
)

print("TOP 10 FILMES DE MAIOR RECEITA COM RANK()")
display(df_p4)

TOP 10 FILMES DE MAIOR RECEITA COM RANK()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao_ranking,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,13913760000.00
2,Avatar: The Way of Water,2320250281.00,11529787696.35
3,AVENGERS: INFINITY WAR,2052415039.00,10198860811.80
4,spider-man: no way home,1921847111.00,9550042663.98
5,The Lion King,1663075401.00,8264154282.65
6,Top Gun: Maverick,1488732821.00,7397811134.11
7,Barbie,1428545028.00,7098725953.14
8,The Super Mario Bros. Movie,1355725263.00,6736869976.90
9,Black Panther,1349926083.00,6708052691.64
10,Star Wars: The Last Jedi,1332698830.00,6622447026.04


### Pergunta 5
Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos?

In [0]:
data_maxima = (
    spark.table(f"{gold_schema}.dim_movies")
    .filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date()))
    .select(F.max("data_lancamento").alias("max_data"))
    .collect()[0]["max_data"]
)

data_inicio_2anos = (
    spark.table(f"{gold_schema}.dim_movies")
    .filter(F.col("data_lancamento") == data_maxima)
    .select(F.add_months(F.lit(data_maxima), -24).alias("data_limite"))
    .collect()[0]["data_limite"]
)

print(f"Janela de análise: de {data_inicio_2anos} até {data_maxima}")

bridge_person = spark.table(f"{gold_schema}.bridge_movie_person").alias("bp")
people = spark.table(f"{gold_schema}.dim_people").alias("p")

df_p5 = (
    bridge_person
    .join(people.filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id", "inner")
    .join(
        movies.filter(
            (F.col("status_filme") == "Lançado") &
            (F.col("data_lancamento") >= data_inicio_2anos) &
            (F.col("data_lancamento") <= data_maxima)
        ),
        "sk_movie_id",
        "inner"
    )
    .groupBy(F.col("p.nome_pessoa"))
    .agg(F.countDistinct("bp.sk_movie_id").alias("quantidade_participacoes"))
    .orderBy(F.col("quantidade_participacoes").desc())
    .limit(5)
)

print("ATOR COM MAIS FILMES NOS ÚLTIMOS 2 ANOS")
display(df_p5)

Janela de análise: de 2024-02-19 até 2026-02-19
ATOR COM MAIS FILMES NOS ÚLTIMOS 2 ANOS


nome_pessoa,quantidade_participacoes
Kevin Hart,64
Paula Pell,59
John Travolta,59
Melissa Ponzio,59
John Cena,59


### Pergunta 6
Qual produtora de filmes teve o maior Lucro nos últimos 5 anos?

In [0]:
data_inicio_5anos = (
    spark.table(f"{gold_schema}.dim_movies")
    .filter(F.col("data_lancamento") == data_maxima)
    .select(F.add_months(F.lit(data_maxima), -60).alias("data_limite"))
    .collect()[0]["data_limite"]
)

print(f"Janela de análise: de {data_inicio_5anos} até {data_maxima}")

bridge_comp = spark.table(f"{gold_schema}.bridge_movie_company").alias("bc")
companies = spark.table(f"{gold_schema}.dim_companies").alias("c")

df_p6 = (
    bridge_comp
    .join(companies, "sk_company_id", "inner")
    .join(
        movies.filter(
            (F.col("status_filme") == "Lançado") &
            (F.col("data_lancamento") >= data_inicio_5anos) &
            (F.col("data_lancamento") <= data_maxima)
        ),
        "sk_movie_id",
        "inner"
    )
    .join(fact, "sk_movie_id", "inner")
    .groupBy(F.col("c.nome_produtora"))
    .agg(
        F.round(F.sum("f.lucro_brl"), 2).alias("lucro_total_brl"),
        F.round(F.sum("f.lucro_usd"), 2).alias("lucro_total_usd")
    )
    .orderBy(F.col("lucro_total_brl").desc())
    .limit(5)
)

print("PRODUTORA COM MAIOR LUCRO NOS ÚLTIMOS 5 ANOS")
display(df_p6)

Janela de análise: de 2021-02-19 até 2026-02-19
PRODUTORA COM MAIOR LUCRO NOS ÚLTIMOS 5 ANOS


nome_produtora,lucro_total_brl,lucro_total_usd
Universal Pictures,28683860640.89,5772329679.00
Marvel Studios,24614747460.07,4953462823.00
Columbia Pictures,18197462611.75,3662050755.00
Pascal Pictures,13426542134.42,2701952454.00
Illumination,12081881678.03,2431353473.00
